# NM i AI 2026 — Oppgave 1 v3: YOLO + Retrieval
**YOLOv8s (rask trening) + ResNet18 feature matching for klassifisering**

In [ ]:
# CELLE 1: Installer og sjekk GPU
!pip install -q ultralytics==8.1.0
import torch
print(f'GPU: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU - bytt runtime!"}')
!pip show ultralytics | grep Version

In [ ]:
# CELLE 2: Mount Drive og pakk ut data
from google.colab import drive
import zipfile
from pathlib import Path

drive.mount('/content/drive', force_remount=True)

print('Pakker ut train.zip...')
with zipfile.ZipFile('/content/drive/MyDrive/train.zip', 'r') as z:
    z.extractall('/content/')

print('Pakker ut product_images.zip...')
with zipfile.ZipFile('/content/drive/MyDrive/product_images.zip', 'r') as z:
    z.extractall('/content/')

imgs = list(Path('/content/train/images').glob('*.jpg')) + list(Path('/content/train/images').glob('*.jpeg'))
print(f'Treningsbilder: {len(imgs)}')

In [ ]:
# CELLE 3: Konverter COCO -> YOLO labels
import json
from pathlib import Path

with open('/content/train/annotations.json') as f:
    coco = json.load(f)

labels_dir = Path('/content/train/labels')
labels_dir.mkdir(exist_ok=True)

image_info = {img['id']: img for img in coco['images']}
annotations_by_image = {}
for ann in coco['annotations']:
    annotations_by_image.setdefault(ann['image_id'], []).append(ann)

for img_id, img in image_info.items():
    W, H = img['width'], img['height']
    stem = Path(img['file_name']).stem
    lines = []
    for ann in annotations_by_image.get(img_id, []):
        x, y, w, h = ann['bbox']
        xc = max(0.0, min(1.0, (x + w / 2) / W))
        yc = max(0.0, min(1.0, (y + h / 2) / H))
        wn = max(0.0, min(1.0, w / W))
        hn = max(0.0, min(1.0, h / H))
        lines.append(f'{ann["category_id"]} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}')
    (labels_dir / f'{stem}.txt').write_text('\n'.join(lines))

print(f'Konverterte {len(image_info)} label-filer')

In [ ]:
# CELLE 4: Lag dataset.yaml (kun hyllebilder, 90/10 split)
import random, json
from pathlib import Path

all_shelf = sorted(Path('/content/train/images').glob('*.jpg')) + \
            sorted(Path('/content/train/images').glob('*.jpeg'))
random.seed(42)
random.shuffle(all_shelf)
split = int(len(all_shelf) * 0.9)

Path('/content/train_list.txt').write_text('\n'.join(str(p) for p in all_shelf[:split]))
Path('/content/val_list.txt').write_text('\n'.join(str(p) for p in all_shelf[split:]))
print(f'Train: {split}, Val: {len(all_shelf)-split}')

with open('/content/train/annotations.json') as f:
    coco = json.load(f)
cats = sorted(coco['categories'], key=lambda c: c['id'])
yaml_lines = ['path: /content', 'train: train_list.txt', 'val: val_list.txt', '',
              f'nc: {len(cats)}', 'names:']
for c in cats:
    safe = c['name'].replace("'", "").replace('"', '')
    yaml_lines.append(f"  {c['id']}: '{safe}'")
Path('/content/dataset.yaml').write_text('\n'.join(yaml_lines))
print('dataset.yaml lagret!')

In [ ]:
# CELLE 5: Tren YOLOv8s (rask - ~20 min)
import os, torch, functools
os.environ['WANDB_DISABLED'] = 'true'
_orig = torch.load
torch.load = functools.partial(_orig, weights_only=False)

from ultralytics import YOLO
model = YOLO('yolov8s.pt')

model.train(
    data='/content/dataset.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    half=True,
    patience=15,
    name='yolo_detect',
    project='runs',
    exist_ok=True,
)
print('YOLO trening ferdig!')

In [ ]:
# CELLE 6: Ekstraher ResNet18 features for alle produktbilder
import json
import numpy as np
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import functools
from pathlib import Path
from PIL import Image

_orig = torch.load
torch.load = functools.partial(_orig, weights_only=False)

# Last ResNet18 pretrained
resnet = models.resnet18(pretrained=True)
# Fjern siste FC-lag -> 512-dim features
feature_extractor = torch.nn.Sequential(*list(resnet.children())[:-1])
feature_extractor.eval().cuda()

# Lagre ResNet18 weights for submission
Path('/content/submission').mkdir(exist_ok=True)
torch.save(resnet.state_dict(), '/content/submission/resnet18.pth')
print(f'ResNet18 weights lagret: {Path("/content/submission/resnet18.pth").stat().st_size/1024/1024:.1f} MB')

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Match produktnavn -> category_id
with open('/content/train/annotations.json') as f:
    coco = json.load(f)
with open('/content/NM_NGD_product_images/metadata.json') as f:
    meta = json.load(f)

cat_name_to_id = {c['name'].strip().upper(): c['id'] for c in coco['categories']}
code_to_cat = {}
for p in meta['products']:
    name = p['product_name'].strip().upper()
    if name in cat_name_to_id:
        code_to_cat[p['product_code']] = cat_name_to_id[name]
print(f'Matchet {len(code_to_cat)} produkter')

# Ekstraher features
all_features = []
all_labels = []
img_base = Path('/content/NM_NGD_product_images')

with torch.no_grad():
    for product_code, cat_id in code_to_cat.items():
        product_dir = img_base / product_code
        if not product_dir.exists():
            continue
        for img_path in product_dir.glob('*.jpg'):
            try:
                img = Image.open(str(img_path)).convert('RGB')
                tensor = transform(img).unsqueeze(0).cuda()
                feat = feature_extractor(tensor).squeeze().cpu().numpy()
                all_features.append(feat)
                all_labels.append(cat_id)
            except Exception:
                continue

features_arr = np.array(all_features, dtype=np.float32)
labels_arr = np.array(all_labels, dtype=np.int32)

# Normaliser features (for cosine similarity)
norms = np.linalg.norm(features_arr, axis=1, keepdims=True)
features_arr = features_arr / (norms + 1e-8)

np.save('/content/submission/product_features.npy', features_arr)
np.save('/content/submission/product_labels.npy', labels_arr)
print(f'Features shape: {features_arr.shape}')
print(f'Labels shape: {labels_arr.shape}')

In [ ]:
# CELLE 7: Eksporter YOLO til ONNX
!pip install -q onnxscript

import torch, functools
_orig = torch.load
torch.load = functools.partial(_orig, weights_only=False)

from ultralytics import YOLO
model = YOLO('runs/yolo_detect/weights/best.pt')
model.export(format='onnx', opset=17, half=False, imgsz=640)
print('ONNX eksportert!')

import shutil
shutil.copy('runs/yolo_detect/weights/best.onnx', '/content/submission/best.onnx')
print(f'best.onnx: {Path("/content/submission/best.onnx").stat().st_size/1024/1024:.1f} MB')

In [ ]:
# CELLE 8: Lag run.py
run_py = '''
import argparse
import json
import numpy as np
from pathlib import Path
from PIL import Image
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from ultralytics import YOLO


def load_feature_extractor(weights_path):
    resnet = models.resnet18(pretrained=False)
    state = torch.load(str(weights_path), weights_only=False, map_location="cuda")
    resnet.load_state_dict(state)
    extractor = torch.nn.Sequential(*list(resnet.children())[:-1])
    extractor.eval().cuda()
    return extractor


def extract_features(extractor, transform, img_crop):
    with torch.no_grad():
        tensor = transform(img_crop).unsqueeze(0).cuda()
        feat = extractor(tensor).squeeze().cpu().numpy()
    norm = np.linalg.norm(feat)
    return feat / (norm + 1e-8)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", required=True)
    parser.add_argument("--output", required=True)
    args = parser.parse_args()

    input_dir = Path(args.input)
    output_path = Path(args.output)
    base = Path(__file__).parent

    # Last modeller
    yolo = YOLO(str(base / "best.onnx"))
    extractor = load_feature_extractor(base / "resnet18.pth")
    product_features = np.load(str(base / "product_features.npy"))
    product_labels = np.load(str(base / "product_labels.npy"))

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    img_paths = sorted([
        p for p in input_dir.iterdir()
        if p.suffix.lower() in (".jpg", ".jpeg", ".png")
    ])
    print(f"Processing {len(img_paths)} images...")

    predictions = []
    for img_path in img_paths:
        image_id = int(img_path.stem.split("_")[-1])
        img_pil = Image.open(str(img_path)).convert("RGB")
        W, H = img_pil.size

        results = yolo.predict(
            str(img_path),
            device=0,
            conf=0.15,
            iou=0.45,
            verbose=False,
            imgsz=640,
            augment=True,
        )

        for result in results:
            if result.boxes is None:
                continue
            boxes = result.boxes
            for i in range(len(boxes)):
                x1, y1, x2, y2 = boxes.xyxy[i].tolist()
                yolo_cat = int(boxes.cls[i].item())
                conf = float(boxes.conf[i].item())

                # Crop og ekstraher features for retrieval
                x1c = max(0, int(x1))
                y1c = max(0, int(y1))
                x2c = min(W, int(x2))
                y2c = min(H, int(y2))

                if x2c > x1c and y2c > y1c:
                    crop = img_pil.crop((x1c, y1c, x2c, y2c))
                    crop_feat = extract_features(extractor, transform, crop)
                    sims = product_features @ crop_feat
                    best_idx = int(np.argmax(sims))
                    best_sim = float(sims[best_idx])
                    retrieval_cat = int(product_labels[best_idx])
                    # Bruk retrieval hvis similarity er høy nok, ellers YOLO
                    final_cat = retrieval_cat if best_sim > 0.5 else yolo_cat
                else:
                    final_cat = yolo_cat

                predictions.append({
                    "image_id": image_id,
                    "category_id": final_cat,
                    "bbox": [round(x1, 1), round(y1, 1),
                             round(x2 - x1, 1), round(y2 - y1, 1)],
                    "score": round(conf, 3),
                })

    output_path.parent.mkdir(parents=True, exist_ok=True)
    with open(output_path, "w") as f:
        json.dump(predictions, f)
    print(f"Wrote {len(predictions)} predictions to {output_path}")


if __name__ == "__main__":
    main()
'''.strip()

from pathlib import Path
Path('/content/submission/run.py').write_text(run_py)
print('run.py lagret!')

In [ ]:
# CELLE 9: Lag ZIP og last ned
import zipfile
from pathlib import Path

sub_dir = Path('/content/submission')
files = list(sub_dir.iterdir())
print('Filer i submission:')
for f in files:
    print(f'  {f.name}: {f.stat().st_size/1024/1024:.1f} MB')

zip_path = '/content/submission_mathea_v3.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in files:
        zf.write(str(f), f.name)

zip_mb = Path(zip_path).stat().st_size / 1024 / 1024
print(f'\nTotal ZIP: {zip_mb:.1f} MB (maks 420 MB)')

from google.colab import files
files.download(zip_path)
print('Last opp submission_mathea_v3.zip på app.ainm.no/submit/norgesgruppen-data')